# Cheaper model evaluation without IRT

## Predict selectively. Audit randomly. Rank defensibly.

**MMLU + SWE-bench Verified · held-out replay · August 2026**

> The central result: prediction can concentrate evaluation effort, while a randomized audit protects the benchmark score when prediction is wrong.

---

# The 30-second version

- We built a **non-IRT** path for evaluating only a subset of benchmark tasks.
- A historical response model predicts the unobserved tasks.
- Randomized sampling with logged probabilities corrects prediction error.
- Models share selected tasks, enabling lower-noise paired comparisons.
- On held-out MMLU models, **5% of tasks produced median Kendall τ = 0.916**.
- On later SWE-bench systems, a sequential design improved the 5% result from **0.617 to 0.656**.

### Bottom line

The evaluation architecture works. Better SWE-specific prediction is the next multiplier.

---

# The problem

A full evaluation spends equally on:

- highly redundant tasks;
- tasks that do not affect the ranking decision;
- model pairs that are already clearly separated;
- and uncertain comparisons that actually need more evidence.

## Our question

**Can we buy fewer observations without turning a prediction model into the ground truth?**

---

# The architecture

| Stage | Action | Purpose |
|---|---|---|
| 1 · Learn | Fit a low-rank response surrogate on completed historical models | Capture reusable cross-task structure |
| 2 · Probe | Evaluate a content-stratified sentinel set | Adapt predictions to the new model |
| 3 · Focus | Select tasks with high predicted information value | Spend on observations likely to matter |
| 4 · Audit | Preserve randomized exploration and log inclusion probabilities | Correct surrogate error |
| 5 · Rank | Estimate shared-item model gaps and uncertainty | Resolve only supported comparisons |

### No IRT dependency

The predictor can later be replaced with embeddings, nearest neighbors, matrix completion, or an ensemble. The correction layer remains the same.

---

# Why the score remains defensible

For task outcome $y_i$, prediction $q_i$, and logged inclusion probability $\pi_i$:

$$
\widehat{\mu}
= \frac{1}{N}\sum_i q_i
+ \frac{1}{N}\sum_{i\in S}\frac{y_i-q_i}{\pi_i}
$$

The first term predicts the full benchmark. The second term audits and corrects that prediction.

## The key property

**A weak surrogate increases variance; it does not redefine the benchmark score.**

---

# Evidence design

| | MMLU | SWE-bench Verified |
|---|---:|---:|
| Historical rows used for training | 296 models | 100 systems |
| Held-out evaluation rows | 99 models | 34 later systems |
| Tasks | 14,042 | 500 |
| Content strata | 57 subjects | 12 repositories |
| Split | Entire creator groups held out | Chronologically latest quarter held out |
| Replicates | 10 seeds | 10 seeds |

Outcomes remained hidden until an item was selected. Every candidate model used the same realized item set for paired comparison.

---

# Result 1 · Strong MMLU ranking at low cost

| Budget | Median tasks/model | Median Kendall τ | Median score MAE | Median interval coverage |
|---:|---:|---:|---:|---:|
| **1%** | 144 | **0.8085** | 0.0296 | 95.5% |
| **5%** | 707 | **0.9157** | 0.0119 | 96.0% |
| **10%** | 1,397 | **0.9404** | 0.0083 | 95.5% |

## The presentation headline

**Using 5% of MMLU, the method recovered the full ranking with τ ≈ 0.92 while maintaining approximately nominal score coverage.**

---

# Result 2 · Sequential evaluation helped SWE at 5%

The statistically safe sequential design reserves its final round for a randomized audit.

| SWE budget | One-shot τ | Sequential τ | Outcome |
|---:|---:|---:|---|
| 2% · about 9 tasks | 0.5268 | **0.5489** | Sequential improvement |
| **5% · about 24 tasks** | 0.6169 | **0.6559** | **Largest useful gain** |
| 10% · about 50 tasks | **0.6955** | 0.6691 | One-shot remains preferable |

At 5%, sequential evaluation gained **0.039 Kendall τ** while retaining median interval coverage of approximately **95.6%**.

---

# What specifically worked

### 1. Historical responses contained reusable structure
Enough signal transferred to rank completely held-out models and later systems.

### 2. Randomized correction protected the estimand
We did not need to assume that the surrogate was calibrated or structurally correct.

### 3. Shared tasks strengthened comparisons
Paired gaps use the same issue outcomes instead of comparing two unrelated samples.

### 4. Sequential learning can help in the lowest-budget regime
On SWE-bench, the intermediate update improved ordering when only about 24 tasks were available.

### 5. The implementation is lightweight
The full replay completed in under 20 seconds; **53 tests**, Ruff, and strict mypy pass.

---

# Correct ranking means allowing ties

A cheap evaluation should not force a total order when the evidence cannot resolve one.

## Decision rule

Declare model A above model B only when the lower confidence bound for their paired gap exceeds the practical threshold $\epsilon$.

Otherwise:

- keep the pair in the same unresolved tier; or
- buy another shared batch targeted at that comparison.

> The display order is convenient. The confidence-backed partial order is the claim.

---

# Why SWE-bench remains harder

- Only 100 historical systems were available to predict 34 later systems.
- The 500 issues span 12 repositories with different code and failure modes.
- Rows combine the model, agent scaffold, tools, and inference policy.
- The chronological holdout creates real distribution shift.

## Productive interpretation

The randomized correction is doing its job. The next bottleneck is the **SWE-specific surrogate**, not the validity architecture.

---

# The next efficiency gain

Upgrade the surrogate with information the current SVD cannot see:

1. Repository and task-difficulty effects.
2. Issue-text and codebase embeddings.
3. Model-family and agent-scaffold features.
4. Pairwise acquisition focused on unresolved leaderboard neighbors.
5. Fixed-size stratified audit sampling to reduce propensity-weight variance.

### What stays unchanged

Randomized auditing, logged probabilities, paired gaps, and unresolved confidence tiers.

---

# Questions you are likely to get

<details><summary><b>Does this depend on IRT?</b></summary><p>No. The predictor is low-rank matrix regression and the final estimator is design-based. IRT remains a separate optional ranking path.</p></details>

<details><summary><b>What if the surrogate is wrong?</b></summary><p>The randomized audit estimates and corrects its residual error. A poor surrogate costs efficiency, not a different target score.</p></details>

<details><summary><b>Is 5% of SWE-bench enough for a definitive leaderboard?</b></summary><p>Not yet. It supports useful coarse ordering, but close pairs remain unresolved and should receive more shared tasks.</p></details>

<details><summary><b>Why did sequential lose at 10%?</b></summary><p>The extra low-rank fitting observations did not repay the randomized audit budget they consumed. That motivates a richer repository- and scaffold-aware predictor.</p></details>

<details><summary><b>Are SWE rows rankings of base models?</b></summary><p>No. They rank model-plus-agent systems under the submitted scaffold and evaluation conditions.</p></details>

---

# Closing

## What we demonstrated

A non-IRT evaluation system can:

- concentrate evaluation effort using historical response structure;
- correct prediction errors through randomized auditing;
- estimate shared-item model gaps;
- and recover useful rankings from a fraction of the benchmark.

# Predict selectively. Audit randomly. Rank defensibly.

---

# Appendix · Evidence and reproduction

- Detailed methodology: [`docs/NON_IRT_EVALUATION.md`](docs/NON_IRT_EVALUATION.md)
- HTML evidence report: [`REPORT.html`](REPORT.html)
- Implementation: [`src/irt_rank/efficient.py`](src/irt_rank/efficient.py)
- Machine-readable results: [non-IRT replay artifact](.agent/ml/adaptive-irt-ranking/artifacts/non-irt-efficiency-result.json)

```bash
uv run python scripts/run_non_irt_efficiency.py
uv run pytest
uv run ruff check .
uv run mypy src scripts tests
```